In [16]:

import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin


class BatchGD(RegressorMixin, BaseEstimator):

    def __init__(self, learning_rate=0.01, epochs=1000):
        self.learning_rate = learning_rate
        self.epochs = epochs

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        y = np.asarray(y, dtype=float).reshape(-1)

        if X.ndim == 1:
            X = X.reshape(-1, 1)

        if X.ndim != 2 or X.shape[0] == 0:
            raise ValueError("X must be a non-empty 2D array.")

        if X.shape[0] != len(y):
            raise ValueError("X and y must have the same number of samples.")

        if not np.all(np.isfinite(X)) or not np.all(np.isfinite(y)):
            raise ValueError("X and y must contain finite values.")

        n_samples, n_features = X.shape

        # Initialize parameters
        self.w_ = np.zeros(n_features, dtype=float)
        self.b_ = 0.0
        self.n_features_in_ = n_features
        self.loss_history_ = []

        # Batch Gradient Descent
        for epoch in range(self.epochs):

            y_pred = X @ self.w_ + self.b_
            error = y_pred - y

            dw = (2 / n_samples) * (X.T @ error)
            db = (2 / n_samples) * np.sum(error)

            self.w_ -= self.learning_rate * dw
            self.b_ -= self.learning_rate * db

            # Record mean squared error
            updated_pred = X @ self.w_ + self.b_
            loss = np.mean((updated_pred - y) ** 2)
            self.loss_history_.append(loss)

        return self

    def predict(self, X):
        if not hasattr(self, "w_"):
            raise ValueError("Call fit() before predict().")

        X = np.asarray(X, dtype=float)

        if X.ndim == 1:
            if X.size != self.n_features_in_:
                raise ValueError("Input has the wrong number of features.")
            X = X.reshape(1, -1)

        if X.ndim != 2 or X.shape[1] != self.n_features_in_:
            raise ValueError("X has the wrong number of features.")

        return X @ self.w_ + self.b_


In [14]:
import pandas as pd
X = pd.DataFrame(np.arange(1, 10, dtype=float),columns=['x'])
y = 4 * X + 3

model = BatchGD(
    learning_rate=0.0001,
    epochs=5000
)

model.fit(X, y)

print("Learned weight:", model.w)
print("Learned bias:", model.b)
print("Initial loss:", model.loss_history[0])
print("Final loss:", model.loss_history)

print("Predictions:", model.predict([1, 5, 10]))


Learned weight: [4.29844803]
Learned bias: 1.1220877877066802
Initial loss: 627.4540346785185
Final loss: [np.float64(627.4540346785185), np.float64(619.3476924139076), np.float64(611.3462642322013), np.float64(603.4483922968141), np.float64(595.6527363447822), np.float64(587.9579734593174), np.float64(580.3627978453084), np.float64(572.8659206077248), np.float64(565.4660695328897), np.float64(558.1619888725842), np.float64(550.9524391309455), np.float64(543.8361968541218), np.float64(536.8120544226522), np.float64(529.8788198465305), np.float64(523.0353165629248), np.float64(516.2803832365121), np.float64(509.6128735623986), np.float64(503.03165607159036), np.float64(496.53561393898224), np.float64(490.12364479383007), np.float64(483.7946605326779), np.float64(477.54758713470517), np.float64(471.38136447946454), np.float64(465.2949461669771), np.float64(459.2872993401583), np.float64(453.35740450953955), np.float64(447.50425538026093), np.float64(441.7268586813004), np.float64(436.024

In [17]:

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score, mean_absolute_error

# Load dataset
housing = fetch_california_housing(as_frame=True)

X = housing.data.to_numpy()
y = housing.target.to_numpy()

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Custom gradient descent pipeline
pipeline1 = make_pipeline(
    StandardScaler(),
    BatchGD(learning_rate=0.01, epochs=1000)
)

# Scikit-learn linear regression pipeline
pipeline2 = make_pipeline(
    StandardScaler(),
    LinearRegression()
)

# Train both models
pipeline1.fit(X_train, y_train)
pipeline2.fit(X_train, y_train)

# Predict
pred1 = pipeline1.predict(X_test)
pred2 = pipeline2.predict(X_test)

# Evaluate
print("BatchGD R2:", r2_score(y_test, pred1))
print("LinearRegression R2:", r2_score(y_test, pred2))

print("BatchGD MAE:", mean_absolute_error(y_test, pred1))
print("LinearRegression MAE:", mean_absolute_error(y_test, pred2))


BatchGD R2: 0.5767889905063643
LinearRegression R2: 0.575787706032451
BatchGD MAE: 0.5352068799388349
LinearRegression MAE: 0.5332001304956565
